# 01 · Explore the map

Get to know the reference raster before writing any matching code.

`data/dresden-map.png` is 8192 x 5074 (41.6 MPix), **RGBA** — `cv2.imread` drops alpha by default, which is usually what you want, but be deliberate about it.

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from navmod import matcher, camera, metrics, georef
from navmod.augment import AugmentConfig, augment_frame

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

rng = np.random.default_rng()


def show(img, title=None, figsize=(11, 7)):
    """Inline display for OpenCV arrays (handles BGR->RGB and grayscale)."""
    plt.figure(figsize=figsize)
    gray = img.ndim == 2
    plt.imshow(img if gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
               cmap="gray" if gray else None)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


print("ready ·", cv2.__version__)

## Load it

Check dtype, shape and channel count. Decide once whether you work in colour or grayscale and stay consistent — detectors want grayscale anyway.

## Establish scale

What ground distance does the map span? That fixes your GSD (metres/pixel) and makes every later error number physically meaningful. Good first use of `metrics.gsd`.

If it is 5 km across 8192 px, GSD is about 0.61 m/px.

## Look at crops

Pull a few 512x512 windows from different regions — dense city, river, parkland, farmland. These are what your drone will actually see.

## Where are the features?

Run a detector over several crops and count keypoints. Feature-poor regions (water, uniform fields) are where localization will fail, and knowing *which* parts of your map are weak will explain later results. Consider a coarse heatmap of keypoint density over the whole map.

## estimate_pose on Dresden map

Render synthetic frames at known positions, run `estimate_pose`, visualise position error and heading on the map.

In [ ]:
from navmod.matcher import estimate_pose
from navmod.georef import MapGeoref, px_to_latlon, latlon_to_px, distance_m
from navmod.camera import render_frame
from navmod.types import Fix

map_img = cv2.imread(str(DATA / "dresden-map.png"), cv2.IMREAD_COLOR)
print(f"map: {map_img.shape}")

geo = MapGeoref(
    top_left_lat=51.0800079, top_left_lon=13.6911106,
    bottom_right_lat=51.0575373, bottom_right_lon=13.7485796,
    image_width_px=map_img.shape[1], image_height_px=map_img.shape[0],
)
print(f"GSD: {geo.gsd_x:.3f} x {geo.gsd_y:.3f} m/px")

# Augmentation presets
aug_mild = AugmentConfig(
    brightness=10, contrast=1.08, noise_sigma=8, blur_ksize=3,
    jpeg_quality=90, haze_alpha=0.05, vignette_strength=0.2,
    gamma=1.1, color_shift=(3, -2, -5),
)
aug_default = AugmentConfig()
aug_harsh = AugmentConfig(
    brightness=25, contrast=1.25, noise_sigma=20, blur_ksize=5,
    jpeg_quality=65, haze_alpha=0.15, vignette_strength=0.5,
    gamma=1.3, color_shift=(10, -8, -15),
)

### Single frame — clean vs augmented at a random location

In [ ]:
h, w = map_img.shape[:2]
margin = 400

# Random location every run
true_px = (int(rng.integers(margin, w - margin)),
           int(rng.integers(margin, h - margin)))
heading_deg = float(rng.integers(0, 360))
scale = float(rng.choice([1.0, 1.5, 2.0]))

print(f"Location: px={true_px}, heading={heading_deg:.0f}°, scale={scale}")

frame_clean = render_frame(map_img, *true_px, heading_deg=heading_deg, scale=scale)
frame_aug   = render_frame(map_img, *true_px, heading_deg=heading_deg, scale=scale,
                           augment=aug_default)

fix_clean = estimate_pose(frame_clean, map_img, geo)
fix_aug   = estimate_pose(frame_aug, map_img, geo)

true_lat, true_lon = px_to_latlon(geo, *true_px)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Clean frame
axes[0].imshow(cv2.cvtColor(frame_clean, cv2.COLOR_BGR2RGB))
if fix_clean.ok:
    err_c = distance_m(fix_clean.lat, fix_clean.lon, true_lat, true_lon)
    axes[0].set_title(f"Clean — err {err_c:.1f} m, σ {fix_clean.sigma_m:.1f} m")
else:
    axes[0].set_title("Clean — NO FIX")
axes[0].axis("off")

# Augmented frame
axes[1].imshow(cv2.cvtColor(frame_aug, cv2.COLOR_BGR2RGB))
if fix_aug.ok:
    err_a = distance_m(fix_aug.lat, fix_aug.lon, true_lat, true_lon)
    axes[1].set_title(f"Augmented — err {err_a:.1f} m, σ {fix_aug.sigma_m:.1f} m")
else:
    axes[1].set_title("Augmented — NO FIX")
axes[1].axis("off")

# Map crop with true vs estimated positions
crop_r = 600
cx, cy = true_px
y0, y1 = max(0, cy - crop_r), min(h, cy + crop_r)
x0, x1 = max(0, cx - crop_r), min(w, cx + crop_r)
crop = map_img[y0:y1, x0:x1]
axes[2].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
axes[2].plot(cx - x0, cy - y0, "g+", ms=20, mew=3, label="true")
if fix_clean.ok:
    ep = latlon_to_px(geo, fix_clean.lat, fix_clean.lon)
    axes[2].plot(ep[0] - x0, ep[1] - y0, "b^", ms=12, mew=2, label=f"clean ({err_c:.0f} m)")
if fix_aug.ok:
    ep = latlon_to_px(geo, fix_aug.lat, fix_aug.lon)
    axes[2].plot(ep[0] - x0, ep[1] - y0, "rx", ms=15, mew=3, label=f"aug ({err_a:.0f} m)")
axes[2].legend(fontsize=10)
axes[2].set_title("Map: true (green) vs estimates")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### Grid test — clean vs augmented error maps

In [ ]:
grid_step = 800
margin = 300
hdg, sc = 0, 1.0

h, w = map_img.shape[:2]
xs = list(range(margin, w - margin, grid_step))
ys = list(range(margin, h - margin, grid_step))

results_clean = []
results_aug = []

for px_x in xs:
    for px_y in ys:
        tlat, tlon = px_to_latlon(geo, px_x, px_y)

        frame_c = render_frame(map_img, px_x, px_y, heading_deg=hdg, scale=sc)
        fix_c = estimate_pose(frame_c, map_img, geo)
        err_c = distance_m(fix_c.lat, fix_c.lon, tlat, tlon) if fix_c.ok else np.nan
        results_clean.append((px_x, px_y, err_c, fix_c.sigma_m, fix_c.ok))

        frame_a = render_frame(map_img, px_x, px_y, heading_deg=hdg, scale=sc,
                               augment=aug_default)
        fix_a = estimate_pose(frame_a, map_img, geo)
        err_a = distance_m(fix_a.lat, fix_a.lon, tlat, tlon) if fix_a.ok else np.nan
        results_aug.append((px_x, px_y, err_a, fix_a.sigma_m, fix_a.ok))

results_clean = np.array(results_clean)
results_aug = np.array(results_aug)

for label, res in [("Clean", results_clean), ("Augmented", results_aug)]:
    n_ok = int(np.sum(res[:, 4]))
    errs = res[res[:, 4] == 1, 2]
    med = np.nanmedian(errs) if len(errs) else float("nan")
    mx = np.nanmax(errs) if len(errs) else float("nan")
    print(f"{label:10s}: {n_ok}/{len(res)} fixes | median {med:.1f} m | max {mx:.1f} m")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
thumb = cv2.cvtColor(cv2.resize(map_img, (w // 4, h // 4)), cv2.COLOR_BGR2RGB)

for col, (label, res) in enumerate([("Clean", results_clean), ("Augmented", results_aug)]):
    ok_mask = res[:, 4] == 1
    errs = res[ok_mask, 2]
    vmax = max(30, np.nanpercentile(errs, 95)) if len(errs) else 30

    axes[0, col].imshow(thumb, extent=[0, w, h, 0])
    if ok_mask.any():
        sc_ = axes[0, col].scatter(res[ok_mask, 0], res[ok_mask, 1],
                                    c=res[ok_mask, 2], cmap="RdYlGn_r", s=80,
                                    vmin=0, vmax=vmax, edgecolors="k", linewidths=0.5)
        plt.colorbar(sc_, ax=axes[0, col], label="error (m)")
    fail_mask = res[:, 4] == 0
    if fail_mask.any():
        axes[0, col].scatter(res[fail_mask, 0], res[fail_mask, 1],
                             c="black", s=80, marker="x", linewidths=2, label="no fix")
        axes[0, col].legend()
    axes[0, col].set_title(f"{label}: position error")
    axes[0, col].axis("off")

    # Sigma vs actual error
    if ok_mask.any():
        axes[1, col].scatter(res[ok_mask, 3], res[ok_mask, 2], alpha=0.6, s=40)
        lim = max(np.nanmax(errs), np.nanmax(res[ok_mask, 3])) * 1.1
        axes[1, col].plot([0, lim], [0, lim], "k--", lw=1, label="σ = error")
        axes[1, col].set_xlabel("reported σ (m)")
        axes[1, col].set_ylabel("actual error (m)")
        axes[1, col].legend()
    axes[1, col].set_title(f"{label}: uncertainty calibration")

plt.tight_layout()
plt.show()

### Heading & scale sweep — clean vs augmented at a random point

In [ ]:
cx = int(rng.integers(margin, w - margin))
cy = int(rng.integers(margin, h - margin))
print(f"Sweep centre: px=({cx}, {cy})")

headings = np.arange(0, 360, 15)
scales = [1.0, 1.5, 2.0, 3.0]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (mode, aug) in enumerate([("Clean", False), ("Augmented", aug_default)]):
    for sc in scales:
        pos_errs, hdg_errs = [], []
        for hdg in headings:
            frame = render_frame(map_img, cx, cy, heading_deg=hdg, scale=sc,
                                 augment=aug)
            fix = estimate_pose(frame, map_img, geo)
            tlat, tlon = px_to_latlon(geo, cx, cy)
            if fix.ok:
                pos_errs.append(distance_m(fix.lat, fix.lon, tlat, tlon))
                diff = np.degrees(fix.heading_rad) - hdg
                diff = (diff + 180) % 360 - 180
                hdg_errs.append(abs(diff))
            else:
                pos_errs.append(np.nan)
                hdg_errs.append(np.nan)
        axes[row, 0].plot(headings, pos_errs, ".-", label=f"scale={sc}")
        axes[row, 1].plot(headings, hdg_errs, ".-", label=f"scale={sc}")

    axes[row, 0].set_xlabel("true heading (°)")
    axes[row, 0].set_ylabel("position error (m)")
    axes[row, 0].set_title(f"{mode}: position error vs heading")
    axes[row, 0].legend()
    axes[row, 0].set_ylim(bottom=0)

    axes[row, 1].set_xlabel("true heading (°)")
    axes[row, 1].set_ylabel("heading error (°)")
    axes[row, 1].set_title(f"{mode}: heading error vs heading")
    axes[row, 1].legend()
    axes[row, 1].set_ylim(bottom=0)

plt.tight_layout()
plt.show()

### Augmentation gallery — visual comparison

Side-by-side: raw map crop vs mild / default / harsh augmentation at a random spot.

In [ ]:
gx = int(rng.integers(margin, w - margin))
gy = int(rng.integers(margin, h - margin))
print(f"Gallery location: px=({gx}, {gy})")

presets = [
    ("Raw crop",  False),
    ("Mild",      aug_mild),
    ("Default",   aug_default),
    ("Harsh",     aug_harsh),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (label, aug) in zip(axes, presets):
    frame = render_frame(map_img, gx, gy, heading_deg=0, scale=1.0, augment=aug)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontsize=13)
    ax.axis("off")
plt.suptitle("Same location — increasing augmentation", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()